# Module 9: Honest Inference with Few Clusters

*Developed by Yin Zhang, PhD, Assistant Professor, Department of Mathematics and Statistics, Washington State University. Part of the Public Safety Statistics Tutorials, developed for WADEPS through CISER.*

---

Every interval so far has rested on asymptotics. **Eleven agencies do not
supply them.**

This module puts four different intervals on the same estimate, finds that
the one most people reach for is the least honest, and shows the two that do
not need large sample theory.

**About 30 minutes.**

## 1. Setup

In [ ]:
# Where the data lives.
#   On Google Colab this reads straight from GitHub.
#   Running from inside a local clone of the repository also works.
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

GITHUB = "https://raw.githubusercontent.com/OWNER/REPO/main/Data/"
_local = Path("../../../Data")
BASE = f"{_local}/" if _local.exists() else GITHUB

monthly = pd.read_csv(BASE + "agency_monthly.csv")
profile = pd.read_csv(BASE + "agency_profile.csv")

# The five agencies that adopted the de escalation training in July 2023.
TRAINED = ["A001", "A002", "A004", "A007", "A010"]
TRUTH = -12.0                       # the effect built into the data, in percent

f = monthly[monthly["provisional"] == 0].copy()          # drop the unfinished months
f = f[~((f["agency_id"] == "A002") & (f["year_month"] == "2021-06"))]   # documented unrest
f["trained"] = f["agency_id"].isin(TRAINED).astype(int)

# Three periods, not two. The program phased in between July and November 2023.
f["period"] = np.where(f["year_month"] >= "2023-11", "after",
                       np.where(f["year_month"] < "2023-07", "before", "phase"))

NAME = dict(zip(profile["agency_id"], profile["agency_name"]))
COMPARISON = sorted(a for a in f["agency_id"].unique() if a not in TRAINED)


def rate(d):
    """Use of force per 100 arrests, pooled over whatever rows are passed in."""
    return 100 * d["n_uof"].sum() / d["n_arrests"].sum()


def cell_rate(agencies, period):
    return rate(f[f["agency_id"].isin(agencies) & (f["period"] == period)])


print(f"{f['agency_id'].nunique()} agencies, {f['year_month'].nunique()} months")
print(f"trained: {', '.join(NAME[a].split()[0] for a in TRAINED)}")

In [ ]:
import statsmodels.api as sm
import statsmodels.formula.api as smf

KEEP = [a for a in TRAINED if a != "A007"]     # the pre trend violator, Intermediate 8
BASELINE = profile.set_index("agency_id")["pre_program_uof_per_100_arrests"]

d = f[f["agency_id"] != "A007"].copy()
d["lo"] = np.log(d["n_arrests"])
pi = pd.PeriodIndex(d["year_month"], freq="M")
d["yr"] = pi.year.values + (pi.month.values - 1) / 12.0
d["base"] = d["agency_id"].map(BASELINE)

pct = lambda b: 100 * (np.exp(b) - 1)


def fit(data, treated, form=None, outcome="n_uof", offset=None):
    s = data.copy()
    s["settled"] = ((s["agency_id"].isin(treated))
                    & (s["period"] == "after")).astype(float)
    s["phase"] = ((s["agency_id"].isin(treated))
                  & (s["period"] == "phase")).astype(float)
    fo = form or f"{outcome} ~ C(agency_id)+C(year_month)+settled+phase"
    z = smf.glm(fo, s, family=sm.families.Poisson(),
                offset=s["lo"] if offset is None else offset).fit()
    lo, hi = z.conf_int().loc["settled"]
    return pct(z.params["settled"]), pct(lo), pct(hi), z

In [ ]:
d["settled"] = ((d["agency_id"].isin(KEEP)) & (d["period"] == "after")).astype(float)
d["phase"] = ((d["agency_id"].isin(KEEP)) & (d["period"] == "phase")).astype(float)
FORM = "n_uof ~ C(agency_id)+C(year_month)+settled+phase"
z = smf.glm(FORM, d, family=sm.families.Poisson(), offset=d["lo"]).fit()
real = pct(z.params["settled"])
print(f"  the estimate: {real:+.2f}%   from {d['agency_id'].nunique()} agencies")

## 2. Model based and cluster robust

In [ ]:
zc = smf.glm(FORM, d, family=sm.families.Poisson(), offset=d["lo"]).fit(
    cov_type="cluster", cov_kwds={"groups": d["agency_id"]})
for lab, zz in [("model based", z), ("cluster robust", zc)]:
    lo, hi = zz.conf_int().loc["settled"]
    print(f"  {lab:16s} se {zz.bse['settled']:.4f}   "
          f"[{pct(lo):+.1f}, {pct(hi):+.1f}]")
print(f"\n  clusters: {d['agency_id'].nunique()}")

The two are **identical to four decimal places**, and that is the warning
rather than the reassurance.

Cluster robust standard errors are consistent as the number of clusters grows.
With eleven they are known to be biased downward, often substantially, and the
usual guidance is that forty or more are needed before they can be trusted.
Agreeing with the model based errors does not mean both are right; it means
neither correction is doing anything.

## 3. The cluster bootstrap

Resample whole agencies with replacement, refit, and read the percentiles.
This makes no asymptotic claim about the number of clusters.

In [ ]:
rng = np.random.default_rng(21)
ids = sorted(d["agency_id"].unique())
boots = []
for _ in range(400):
    pick = rng.choice(ids, len(ids), replace=True)
    s = pd.concat([d[d["agency_id"] == a].assign(agency_id=f"{a}_{i}")
                   for i, a in enumerate(pick)])
    try:
        zz = smf.glm(FORM, s, family=sm.families.Poisson(), offset=s["lo"]).fit()
        boots.append(pct(zz.params["settled"]))
    except Exception:
        pass
boots = np.array(boots)
print(f"  cluster bootstrap over {len(boots)} resamples")
print(f"    95 percent interval [{np.percentile(boots, 2.5):+.1f}, "
      f"{np.percentile(boots, 97.5):+.1f}]")
print(f"    width {np.percentile(boots, 97.5) - np.percentile(boots, 2.5):.1f} points, "
      f"against {pct(z.conf_int().loc['settled'][1]) - pct(z.conf_int().loc['settled'][0]):.1f} "
      f"for the model based interval")

The bootstrap interval is **wider**, 12.9 points against 11.0, and wider on
the side that matters: its lower end reaches 20.9 percent against the model
based 17.9.

The model based interval was too narrow by about a sixth. That is not a
catastrophe and it is not nothing, and **it is available for the price of four
hundred refits.**

## 4. Randomisation inference

A different question, and for a design like this one a better one: among all
the ways four agencies could have been labelled treated, how unusual is the
one that was?

In [ ]:
fakes = []
for _ in range(400):
    pick = list(rng.choice(ids, len(KEEP), replace=False))
    s = d.copy()
    s["settled"] = ((s["agency_id"].isin(pick)) & (s["period"] == "after")).astype(float)
    s["phase"] = ((s["agency_id"].isin(pick)) & (s["period"] == "phase")).astype(float)
    zz = smf.glm(FORM, s, family=sm.families.Poisson(), offset=s["lo"]).fit()
    fakes.append(pct(zz.params["settled"]))
fakes = np.array(fakes)
print(f"  the null distribution over 400 reassignments")
print(f"    median {np.median(fakes):+.1f}%, "
      f"95 percent range [{np.percentile(fakes, 2.5):+.1f}, "
      f"{np.percentile(fakes, 97.5):+.1f}]")
print(f"\n  the real estimate {real:+.1f}% sits at p = "
      f"{np.mean(fakes <= real):.3f}")

This assumes nothing about sample size or the shape of a sampling
distribution. It asks only whether the observed labelling produces an unusual
answer among the labellings that were possible.

**It agrees with the other three about the conclusion**, which was not
guaranteed and is worth reporting as a fact rather than assuming.

Note what it is not: an interval for the effect. The range printed is the
range of the **null**, and inverting a randomisation test to get an interval
requires more machinery than this module covers.

## 5. Which to report

| Method | Assumes | Use when |
|---|---|---|
| Model based | the model, and independence given it | never alone with few clusters |
| Cluster robust | many clusters | 40 or more clusters |
| Wild cluster bootstrap | less than the above | 5 to 40 clusters, the standard advice |
| Cluster bootstrap by resampling | the clusters are exchangeable | few clusters, easy to implement |
| Randomisation inference | the assignment could have been otherwise | few units, a clear assignment mechanism |

**Report at least two, and say which one the conclusion rests on.** If they
disagree, that disagreement is the finding, and with eleven agencies the
model based interval is the one to distrust.

## Exercise

The randomisation test assigned treatment at random among all eleven agencies.
Restrict it to assignments that could plausibly have happened, given that the
program went to high rate agencies.

In [ ]:
# Fill in the blank, then run.
RUN = None          # try True

if RUN:
    top = [a for a in sorted(BASELINE.index, key=lambda x: -BASELINE[x])
           if a in ids][:7]
    rows = []
    for label, pool in [("any four of the eleven", ids),
                        ("any four of the seven highest rate agencies", top)]:
        fk = []
        for _ in range(300):
            pick = list(rng.choice(pool, len(KEEP), replace=False))
            s = d.copy()
            s["settled"] = ((s["agency_id"].isin(pick))
                            & (s["period"] == "after")).astype(float)
            s["phase"] = ((s["agency_id"].isin(pick))
                          & (s["period"] == "phase")).astype(float)
            zz = smf.glm(FORM, s, family=sm.families.Poisson(),
                         offset=s["lo"]).fit()
            fk.append(pct(zz.params["settled"]))
        fk = np.array(fk)
        rows.append({"assignments considered possible": label,
                     "null median": f"{np.median(fk):+.1f}%",
                     "null 95 percent range":
                         f"[{np.percentile(fk, 2.5):+.1f}, {np.percentile(fk, 97.5):+.1f}]",
                     "p for the real estimate": round(float(np.mean(fk <= real)), 3)})
    display(pd.DataFrame(rows).set_index("assignments considered possible"))
else:
    print("Set RUN above, then run this cell again.")

<details>
<summary><b>Solution</b></summary>

```python
RUN = True
```

Restricting the randomisation to plausible assignments shifts the null
distribution and changes the p value.

**That is the central and uncomfortable feature of randomisation inference on
observational data.** The test asks how unusual the observed assignment is
**among a set of assignments you specify**, and specifying that set is a
modelling choice with no data to discipline it. Widening the set toward
implausible assignments makes the null more dispersed and the result look
stronger.

For a genuinely randomised experiment the set is known, and randomisation
inference is exact. Here it is not known, so the honest presentation gives the
p value under a stated assignment set and says what the set was, in the same
way a difference in differences states its comparison group.

The method is still worth running. It simply is not the assumption free
procedure it is sometimes presented as when the randomisation never happened.

</details>

---

**Next:** [Module 10: Propensity Scores and the Overlap Assumption](Module_10_Propensity_Scores.ipynb).

*Part of the Public Safety Statistics Tutorials, developed for the Washington
Data Exchange for Public Safety (WADEPS) through CISER at Washington State
University. Questions or corrections: yin.zhang@wsu.edu*